# SVM Classification: Anime 'Hit' Analysis

This notebook provides a complete pipeline for exploring the MyAnimeList dataset and training an SVM classifier. It uses the refactored modular structure in the `src/` directory.

## 1. Data Loading & Exploration
We use the `src.data_loader` to fetch the dataset.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import os
import sys

# Ensure the local modules are importable
sys.path.append(os.getcwd())

from src.data_loader import load_data
from src.preprocessing import clean_data

# Load and clean data
df_raw = load_data()
df = clean_data(df_raw)
df['target_name'] = df['target'].map({1: 'Hit', 0: 'Standard'})

print(f"Dataset Summary:")
print(df['target_name'].value_counts(normalize=True))

### Visualizing Feature Distributions
We use violin plots to see how feature distributions differ between Hits and Standard releases.

In [ ]:
plt.figure(figsize=(15, 6))
plt.subplot(1, 2, 1)
sns.violinplot(x='target_name', y='members', data=df, palette='muted')
plt.title('Members Distribution vs Hit Status')
plt.yscale('log')

plt.subplot(1, 2, 2)
sns.violinplot(x='target_name', y='popularity', data=df, palette='muted')
plt.title('Popularity Rank Distribution vs Hit Status')
plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

### Multivariate Analysis
Checking the density of Hits using JointPlots.

In [ ]:
sns.jointplot(x='members', y='score', data=df.sample(2000), hue='target_name', alpha=0.5, palette='coolwarm')
plt.show()

## 2. Model Training & Evaluation
Now we use the `src.preprocessing`, `src.train`, and `src.evaluate` modules to build the classification model.

In [ ]:
from src.preprocessing import preprocess_training_data
from src.train import train_and_select_best
from src.evaluate import calculate_metrics, plot_comparison_curves, plot_decision_boundary

# 1. Preprocess
X_train, X_test, y_train, y_test, feature_names = preprocess_training_data(df_raw)

# 2. Train & Select Best Model (PR-AUC)
results, best_kernel = train_and_select_best(X_train, y_train, X_test, y_test, kernels=['rbf', 'poly'])

# 3. Metrics for Best Model
best_results = results[best_kernel]
metrics = calculate_metrics(y_test, best_results['model'].predict(X_test), best_results['y_scores'])

print(f"\n--- Best Model ({best_kernel.upper()}) Metrics ---")
print(f"Accuracy: {metrics['accuracy']:.4f}")
print(f"PR-AUC: {metrics['pr_auc']:.4f}")
print("\nClassification Report:\n", metrics['report'])

### Visualizing Performance
We can plot the curves and decision boundary directly in the notebook.

In [ ]:
# Plot Comparison Curves
plot_comparison_curves(y_test, results, filename='notebook_comparison.png')
plt.show()

# Plot Decision Boundary for the Best Model
plot_decision_boundary(X_test, y_test, kernel=best_kernel, filename='notebook_boundary.png')
plt.show()